In [1]:
import pandas as pd

file_path = 'metadata.csv' 

# low_memory=False --> złożony plik
df = pd.read_csv(file_path, low_memory=False)


### podstawowa analiza
print("Kształt danych (wiersze, kolumny):")
print(df.shape)
print("-" * 30)

print("Pierwsze 5 wierszy:")
print(df.head())
print("-" * 30)

print("Info o typach danych:")
df.info()
print("-" * 30)


print("Kolumny:")
print(df.columns)
print("-" * 30)

print("Liczba brakujących wartości w kolumnach:")
print(df.isnull().sum())

Kształt danych (wiersze, kolumny):
(1056660, 19)
------------------------------
Pierwsze 5 wierszy:
   cord_uid                                       sha source_x  \
0  ug7v899j  d1aafb70c066a2068b02786f8929fd9c900897fb      PMC   
1  02tnwd4m  6b0567729c2143a66d737eb0a2f63f2dce2e5a7d      PMC   
2  ejv2xln0  06ced00a5fc04215949aa72528f2eeaae1d58927      PMC   
3  2b73a28n  348055649b6b8cf2b9a376498df9bf41f7123605      PMC   
4  9785vg6d  5f48792a5fa08bed9f56016f4981ae2ca6031b32      PMC   

                                               title                    doi  \
0  Clinical features of culture-proven Mycoplasma...  10.1186/1471-2334-1-6   
1  Nitric oxide: a pro-inflammatory mediator in l...           10.1186/rr14   
2    Surfactant protein-D and pulmonary host defense           10.1186/rr19   
3               Role of endothelin-1 in lung disease           10.1186/rr44   
4  Gene expression in epithelial cells in respons...           10.1186/rr61   

      pmcid pubmed_id licens

In [2]:
print(f"Rozmiar danych przed czyszczeniem: {df.shape}")

### tylko potrzebne kolumny: title i abstract
### .copy()  -->  nowa niezależna ramka danych
df_clean = df[['title', 'abstract']].copy()

### usunięcie wierszy, w których brak jednoczesnie tytułu i abstraktu
### wiersz zostanie usunięty tylko jeśli wszystkie wartości w podanych kolumnach subset są puste (NaN)
df_clean.dropna(subset=['title', 'abstract'], how='all', inplace=True)

print(f"Rozmiar po usunięciu wierszy bez tytułu i abstraktu: {df_clean.shape}")

### usunięcie duplikatów na podstawie kolumny title
df_clean.drop_duplicates(subset=['title'], inplace=True)

print(f"Rozmiar po usunięciu duplikatów: {df_clean.shape}")


print("\nLiczba brakujących wartości po czyszczeniu:")
print(df_clean.isnull().sum())


print("\nPróbka danych po czyszczeniu:")
print(df_clean.head())

Rozmiar danych przed czyszczeniem: (1056660, 19)
Rozmiar po usunięciu wierszy bez tytułu i abstraktu: (1056268, 2)
Rozmiar po usunięciu duplikatów: (850367, 2)

Liczba brakujących wartości po czyszczeniu:
title            1
abstract    176878
dtype: int64

Próbka danych po czyszczeniu:
                                               title  \
0  Clinical features of culture-proven Mycoplasma...   
1  Nitric oxide: a pro-inflammatory mediator in l...   
2    Surfactant protein-D and pulmonary host defense   
3               Role of endothelin-1 in lung disease   
4  Gene expression in epithelial cells in respons...   

                                            abstract  
0  OBJECTIVE: This retrospective chart review des...  
1  Inflammatory diseases of the respiratory tract...  
2  Surfactant protein-D (SP-D) participates in th...  
3  Endothelin-1 (ET-1) is a 21 amino acid peptide...  
4  Respiratory syncytial virus (RSV) and pneumoni...  


In [3]:
import re

### zastąpienie wartości NaN pustym tekstem ''
df_clean.fillna('', inplace=True)

### nowa połączona kolumna 'text' ### spacja ' ' pomiędzy aby tytuł nie skleił się z abstraktem
df_clean['text'] = df_clean['title'] + ' ' + df_clean['abstract']

### funkcja do czyszczenia tekstu
def preprocess_text(text):
    text = text.lower()  ## zamiana na małe litery
    text = re.sub(r'[^\w\s]', '', text)  ## usunięcie interpunkcji
    text = re.sub(r'\d+', '', text)  ## usunięcie cyfr
    text = re.sub(r'\s+', ' ', text).strip() ## usunięcie nadmiarowych białych znaków
    return text


print("Czyszczenie tekstu...")
df_clean['text_clean'] = df_clean['text'].apply(preprocess_text)
print("Czyszczenie zakończone.")

### 5 wierszy z kolumną oryginalną i wyczyszczoną dla porównania
print("\nPorównanie tekstu oryginalnego i wyczyszczonego:")
print(df_clean[['text', 'text_clean']].head())

Czyszczenie tekstu...
Czyszczenie zakończone.

Porównanie tekstu oryginalnego i wyczyszczonego:
                                                text  \
0  Clinical features of culture-proven Mycoplasma...   
1  Nitric oxide: a pro-inflammatory mediator in l...   
2  Surfactant protein-D and pulmonary host defens...   
3  Role of endothelin-1 in lung disease Endotheli...   
4  Gene expression in epithelial cells in respons...   

                                          text_clean  
0  clinical features of cultureproven mycoplasma ...  
1  nitric oxide a proinflammatory mediator in lun...  
2  surfactant proteind and pulmonary host defense...  
3  role of endothelin in lung disease endothelin ...  
4  gene expression in epithelial cells in respons...  


In [4]:
import spacy
from tqdm import tqdm
import time

### preprocessing z Lematyzacją
nlp = spacy.load('en_core_web_lg', disable=['parser', 'ner']) ## wyłącza komponenty (parser, ner) aby przyspieszyć proces
print("Model załadowany.")

## funkcja do lematyzacji i usuwania stop words
def lemmatize_text(text):
    doc = nlp(text)
    
    # lista lemmatów bez stop words i znaków interpunkcyjnych
    lemmas = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    
    # łączy lemmaty z powrotem w jeden tekst
    return " ".join(lemmas)

## mniejsza próba
print("\nPrzetwarzanie próby 10000 dokumentów...")
df_sample = df_clean.sample(10000, random_state=42)

start_time = time.time()

df_sample_lemmatized = df_sample.copy()
df_sample_lemmatized['text_lemmatized'] = df_sample_lemmatized['text_clean'].apply(lemmatize_text)

end_time = time.time()
print(f"Przetwarzanie próbki zakończone w {(end_time - start_time):.2f} sekund.")


## wyniki
print("\nPorównanie tekstu wyczyszczonego i zlemmatyzowanego:")
print(df_sample_lemmatized[['text_clean', 'text_lemmatized']].head())

Model załadowany.

Przetwarzanie próby 10000 dokumentów...
Przetwarzanie próbki zakończone w 105.69 sekund.

Porównanie tekstu wyczyszczonego i zlemmatyzowanego:
                                               text_clean  \
666494  potent noncovalent inhibitors of the main prot...   
304385  social study resources and social wellbeing be...   
143551  younger adults caught in covid crosshairs as d...   
796106  individuals conceptions of covid pandemic thro...   
283352  profiles of antibody responses against severe ...   

                                          text_lemmatized  
666494  potent noncovalent inhibitor main protease sar...  
304385  social study resource social wellbeing intelli...  
143551  young adult catch covid crosshair demographic ...  
796106  individual conception covid pandemic metaphor ...  
283352  profile antibody response severe acute respira...  


In [10]:
### przetwarzanie (lematyzacja) dla pełnego zbioru (wersja z nlp.pipe)

print("\nRozpoczynam przetwarzanie pełnego zbioru...")
start_time = time.time()


lemmatized_texts = []
### batch_size=500 spaCy przetwarza dokumenty po 500
for doc in tqdm(nlp.pipe(df_clean['text_clean'], batch_size=500), total=len(df_clean)):
    lemmas = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    lemmatized_texts.append(" ".join(lemmas))


df_clean['text_lemmatized'] = lemmatized_texts


end_time = time.time()
total_time_minutes = (end_time - start_time) / 60
print(f"\nPrzetwarzanie pełnego zbioru zakończone w {total_time_minutes:.2f} minut.")


## wyniki
print("\nPorównanie tekstu wyczyszczonego i zlemmatyzowanego:")
print(df_clean[['text_clean', 'text_lemmatized']].head())


Rozpoczynam przetwarzanie pełnego zbioru...


100%|██████████| 850367/850367 [2:18:51<00:00, 102.07it/s]  



Przetwarzanie pełnego zbioru zakończone w 138.86 minut.

Porównanie tekstu wyczyszczonego i zlemmatyzowanego:
                                          text_clean  \
0  clinical features of cultureproven mycoplasma ...   
1  nitric oxide a proinflammatory mediator in lun...   
2  surfactant proteind and pulmonary host defense...   
3  role of endothelin in lung disease endothelin ...   
4  gene expression in epithelial cells in respons...   

                                     text_lemmatized  
0  clinical feature cultureproven mycoplasma pneu...  
1  nitric oxide proinflammatory mediator lung dis...  
2  surfactant proteind pulmonary host defense sur...  
3  role endothelin lung disease endothelin et ami...  
4  gene expression epithelial cell response pneum...  


In [12]:
import joblib

joblib.dump(df_clean, 'df_clean_lemmatized.joblib')
print("DataFrame zapisany w pliku df_clean_lemmatized.joblib")

DataFrame zapisany w pliku df_clean_lemmatized.joblib


In [1]:
import pandas as pd
import spacy
from tqdm import tqdm
import time
import re
import joblib

### wczytywanie obiektu
df_clean = joblib.load('df_clean_lemmatized.joblib')

print("Obiekt wczytany.")

Obiekt wczytany.


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

#### wektoryzacja ze SPECTER
### model 'allenai-specter' specjalnie zaprojektowany do reprezentacji artykułów naukowych
model = SentenceTransformer('allenai-specter', device='cuda')
print("Model załadowany.")

## próbka
n_samples = 20000
df_sample = df_clean.sample(n=n_samples, random_state=42)
texts_to_encode = df_sample['text_lemmatized'].tolist()

### kodowanie tekstów na wektory (embeddings)
start_time = time.time()

embeddings_sample = model.encode(texts_to_encode, 
                                 show_progress_bar=True, ### pokaże pasek postępu
                                 batch_size=512)

end_time = time.time()
print(f"Kodowanie próbki zakończone w {(end_time - start_time):.2f} sekund.")

### wyniki
print("\nKształt macierzy:")
print(embeddings_sample.shape)

### wynik dla próbki
np.save('embeddings_sample.npy', embeddings_sample)
print("Próbka zapisana w pliku 'embeddings_sample.npy'")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Greg Z\.cache\huggingface\hub\models--sentence-transformers--allenai-specter. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model załadowany.


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Kodowanie próbki zakończone w 48.70 sekund.

Kształt macierzy:
(20000, 768)
Próbka zapisana w pliku 'embeddings_sample.npy'


In [5]:
### przetwarzanie pełnego zbioru (SPECTER)

texts_to_encode_full = df_clean['text_lemmatized'].tolist()

start_time = time.time()

embeddings_full = model.encode(texts_to_encode_full, 
                               show_progress_bar=True,
                               batch_size=512)

end_time = time.time()
total_time_minutes = (end_time - start_time) / 60
print(f"\nKodowanie pełnego zbioru zakończone w {total_time_minutes:.2f} minut.")

### wynik
print("\nKształt finalnej macierzy:")
print(embeddings_full.shape)

### zapis
np.save('embeddings_full.npy', embeddings_full)
print("Pełny zbiór zapisany do pliku 'embeddings_full.npy'")

Batches:   0%|          | 0/1661 [00:00<?, ?it/s]


Kodowanie pełnego zbioru zakończone w 33.79 minut.

Kształt finalnej macierzy:
(850367, 768)
Pełny zbiór zapisany do pliku 'embeddings_full.npy'


In [1]:
import pandas as pd
import spacy
from tqdm import tqdm
import time
import re
import joblib
from sentence_transformers import SentenceTransformer
import numpy as np

### wczytywanie obiektów
df_clean = joblib.load('df_clean_lemmatized.joblib')

embeddings_full = np.load('embeddings_full.npy')

print("Obiekty wczytane.")


Obiekty wczytane.


In [3]:
import umap.umap_ as umap

### redukcja wymiarowości z UMAP (z 768 wymiarów do 10)

print(f"Redukcja wymiarowości z 768 do 10 dla {embeddings_full.shape[0]} wektorów...")
start_time = time.time()

### konfiguracja UMAP
## n_neighbors=15 - kontroluje balans między lokalną a globalną strukturą
## n_components=10 - wymiar docelowy, na który redukujemy dane
## min_dist=0.1 - minimalny dystans między punktami (jak gęsto mają być upakowane)
## metric='cosine' - kluczowe dla danych tekstowych, odległość jest mierzona kątem, a nie odległością euklidesową

reducer = umap.UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
    verbose=True
)

### uruchomienie
umap_embeddings = reducer.fit_transform(embeddings_full)

end_time = time.time()
total_time_minutes = (end_time - start_time) / 60
print(f"\nRedukcja wymiarowości zakończona w {total_time_minutes:.2f} minut.")

### wynik i zapis
print("\nKształt macierzy po redukcji UMAP:")
print(umap_embeddings.shape)

np.save('umap_embeddings_10d.npy', umap_embeddings)
print("Osadzenia UMAP zostały zapisane w pliku umap_embeddings_10d.npy")

Redukcja wymiarowości z 768 do 10 dla 850367 wektorów...


O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(angular_rp_forest=True, metric='cosine', n_components=10, n_jobs=1, random_state=42, verbose=True)
Wed Aug 27 21:10:48 2025 Construct fuzzy simplicial set
Wed Aug 27 21:10:48 2025 Finding Nearest Neighbors
Wed Aug 27 21:10:48 2025 Building RP forest with 51 trees
Wed Aug 27 21:11:56 2025 NN descent for 20 iterations
	 1  /  20
	 2  /  20
	 3  /  20
	 4  /  20
	 5  /  20
	Stopping threshold met -- exiting after 5 iterations
Wed Aug 27 21:13:02 2025 Finished Nearest Neighbor Search
Wed Aug 27 21:13:05 2025 Construct embedding


O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Wed Aug 27 21:22:48 2025 Finished embedding

Redukcja wymiarowości zakończona w 12.07 minut.

Kształt macierzy po redukcji UMAP:
(850367, 10)
Osadzenia UMAP zostały zapisane w pliku umap_embeddings_10d.npy


In [1]:
import pandas as pd
import spacy
from tqdm import tqdm
import time
import re
import joblib
from sentence_transformers import SentenceTransformer
import numpy as np

### wczytywanie obiektów
df_clean = joblib.load('df_clean_lemmatized.joblib')

umap_embeddings = np.load('umap_embeddings_10d.npy')

print("Obiekty wczytane.")


Obiekty wczytane.


HDBSCAN

In [4]:
## HDBSCAN

import hdbscan

### konfiguracja HDBSCAN
## min_cluster_size - minimalna liczba dokumentów żeby uznawało za osobny klaster
## min_samples - im wyższa tym więcej punktów uzna za szum
## metric='euclidean' - standardowa metryka dla danych po redukcji UMAP

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=15,
    metric='euclidean',
    cluster_selection_method='eom'
)


start_time = time.time()

### uruchomienie
clusterer.fit(umap_embeddings)


end_time = time.time()
total_time_minutes = (end_time - start_time) / 60
print(f"\nKlastrowanie HDBSCAN zakończone w {total_time_minutes:.2f} minut.")


### dodanie wyników etykiet klastrów do dataframe
df_clean['hdbscan_cluster'] = clusterer.labels_


### wynik znalezionych klastrów (-1 to szum)
n_clusters = len(pd.unique(clusterer.labels_)) - (1 if -1 in clusterer.labels_ else 0)
n_noise = list(clusterer.labels_).count(-1) ### szum

print(f"\nZnaleziono klastrów: {n_clusters}")
print(f"Liczba punktów oznaczonych jako szum: {n_noise} ({n_noise / len(df_clean) * 100:.2f}%)")


print("\nLiczebność 20 największych klastrów (klaster -1 to szum):")

print(df_clean['hdbscan_cluster'].value_counts().head(20))

O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



Klastrowanie HDBSCAN zakończone w 0.94 minut.

Znaleziono klastrów: 492
Liczba punktów oznaczonych jako szum: 504165 (59.29%)

Liczebność 20 największych klastrów (klaster -1 to szum):
hdbscan_cluster
-1      504165
 436      8003
 1        6293
 79       5997
 389      4957
 359      4344
 272      4316
 261      4219
 232      4183
 402      4102
 447      3905
 27       3860
 277      3366
 437      3333
 286      3232
 134      3224
 255      3182
 443      3130
 380      3081
 413      3054
Name: count, dtype: int64


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

### analiza tematyczna największych klastrów HDBSCAN
## pomijając szum

top_clusters_ids = df_clean['hdbscan_cluster'].value_counts().drop(-1).head(5).index.tolist()

print("ANALIZA SŁÓW KLUCZOWYCH DLA NAJWIĘKSZYCH KLASTRÓW")
print("="*30)

for cluster_id in top_clusters_ids:
    cluster_texts = df_clean[df_clean['hdbscan_cluster'] == cluster_id]['text_lemmatized']
    
    ## TF-IDF - kluczowe słowa dla tego klastra
    vectorizer = TfidfVectorizer(max_features=10) ### 10 najważniejszych słów
    vectorizer.fit(cluster_texts)
    
    top_words = vectorizer.get_feature_names_out()
    
    print(f"\nKlaster {cluster_id} (Liczba dokumentów: {len(cluster_texts)})")
    print(f"Słowa kluczowe: {', '.join(top_words)}")

ANALIZA SŁÓW KLUCZOWYCH DLA NAJWIĘKSZYCH KLASTRÓW

Klaster 436 (Liczba dokumentów: 8003)
Słowa kluczowe: case, covid, datum, disease, epidemic, infection, model, number, pandemic, spread

Klaster 1 (Liczba dokumentów: 6293)
Słowa kluczowe: covid, disease, eye, method, ocular, pandemic, patient, result, retinal, study

Klaster 79 (Liczba dokumentów: 5997)
Słowa kluczowe: care, covid, infection, maternal, outcome, pregnancy, pregnant, sarscov, study, woman

Klaster 389 (Liczba dokumentów: 4957)
Słowa kluczowe: analysis, covid, information, medium, model, news, pandemic, sentiment, social, tweet

Klaster 359 (Liczba dokumentów: 4344)
Słowa kluczowe: abstract, covid, editor, editorial, issue, meeting, new, research, time, year


In [8]:
###### dostrajanie HDBSCAN z min_cluster_size=75

### konfiguracja modelu z nowym, niższym progiem
clusterer_v2 = hdbscan.HDBSCAN(
    min_cluster_size=75,
    min_samples=15,
    metric='euclidean',
    cluster_selection_method='eom'
)


start_time = time.time()

### uruchomienie klastrowania
clusterer_v2.fit(umap_embeddings)

end_time = time.time()
total_time_minutes = (end_time - start_time) / 60
print(f"\nKlastrowanie zakończone w {total_time_minutes:.2f} minut.")


### nadpisanie wcześniejszej kolumny z etykietami klastrów
df_clean['hdbscan_cluster'] = clusterer_v2.labels_

### wynik znalezionych klastrów (-1 to szum)
n_clusters = len(pd.unique(clusterer_v2.labels_)) - (1 if -1 in clusterer_v2.labels_ else 0)
n_noise = list(clusterer_v2.labels_).count(-1)

print(f"\nZnaleziono klastrów: {n_clusters}")
print(f"Liczba punktów oznaczonych jako szum: {n_noise} ({n_noise / len(df_clean) * 100:.2f}%)")


print("\nLiczebność 20 największych klastrów (klaster -1 to szum):")

print(df_clean['hdbscan_cluster'].value_counts().head(20))

O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



Klastrowanie zakończone w 0.93 minut.

Znaleziono klastrów: 884
Liczba punktów oznaczonych jako szum: 508360 (59.78%)

Liczebność 20 największych klastrów (klaster -1 to szum):
hdbscan_cluster
-1      508360
 809      8003
 2        6293
 153      5997
 503      4316
 434      4072
 476      4014
 54       3860
 810      3333
 250      3224
 754      3194
 817      3130
 707      3081
 346      2921
 179      2863
 515      2792
 831      2685
 194      2643
 603      2542
 444      2355
Name: count, dtype: int64


In [10]:
### przywracamy pierwszą i najlepszą konfigurację dla HDBSCAN + zapis wyników

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=15,
    metric='euclidean',
    cluster_selection_method='eom'
)


start_time = time.time()

### uruchomienie
clusterer.fit(umap_embeddings)


end_time = time.time()
total_time_minutes = (end_time - start_time) / 60
print(f"\nKlastrowanie HDBSCAN zakończone w {total_time_minutes:.2f} minut.")


### dodanie wyników etykiet klastrów do dataframe
df_clean['hdbscan_cluster'] = clusterer.labels_


### wynik znalezionych klastrów (-1 to szum)
n_clusters = len(pd.unique(clusterer.labels_)) - (1 if -1 in clusterer.labels_ else 0)
n_noise = list(clusterer.labels_).count(-1) ### szum

print(f"\nZnaleziono klastrów: {n_clusters}")
print(f"Liczba punktów oznaczonych jako szum: {n_noise} ({n_noise / len(df_clean) * 100:.2f}%)")


print("\nLiczebność 20 największych klastrów (klaster -1 to szum):")

print(df_clean['hdbscan_cluster'].value_counts().head(20))


### zapis wyników (wersja DataFrame z przypisanymi klastrami HDBSCAN + obiekt wytrenowanego modelu)
joblib.dump(df_clean, 'df_final_hdbscan_clusters.joblib')
print("DataFrame zapisany w df_final_hdbscan_clusters.joblib")


joblib.dump(clusterer, 'hdbscan_model_final.joblib')
print("Model HDBSCAN zapisany w hdbscan_model_final.joblib")

O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



Klastrowanie HDBSCAN zakończone w 0.93 minut.

Znaleziono klastrów: 492
Liczba punktów oznaczonych jako szum: 504165 (59.29%)

Liczebność 20 największych klastrów (klaster -1 to szum):
hdbscan_cluster
-1      504165
 436      8003
 1        6293
 79       5997
 389      4957
 359      4344
 272      4316
 261      4219
 232      4183
 402      4102
 447      3905
 27       3860
 277      3366
 437      3333
 286      3232
 134      3224
 255      3182
 443      3130
 380      3081
 413      3054
Name: count, dtype: int64
DataFrame zapisany w df_final_hdbscan_clusters.joblib
Model HDBSCAN zapisany w hdbscan_model_final.joblib
